# Dataset Preparation

This notebook prepares the mental-health text dataset for the controlled model comparison.

The same fixed train, validation, and test sets will be used for TF-IDF, BiLSTM, and DistilBERT.

## 1. Import libraries

In [1]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

## 2. Load the original dataset

The original `cleaned_mental_health.csv` is kept unchanged.

In [2]:
possible_paths = [
    Path("data/cleaned_mental_health.csv"),
    Path("../data/cleaned_mental_health.csv")
]

DATA_PATH = None

for path in possible_paths:
    if path.exists():
        DATA_PATH = path
        break

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find cleaned_mental_health.csv. "
        "Please check that the dataset is inside the data folder."
    )

df = pd.read_csv(DATA_PATH)

original_rows = len(df)

print("Dataset loaded successfully.")
print("Using:", DATA_PATH)
print("Original shape:", df.shape)

Dataset loaded successfully.
Using: ..\data\cleaned_mental_health.csv
Original shape: (55693, 2)


## 3. Basic validation

In [3]:
print("Columns:", list(df.columns))
print("Missing values:")
print(df[["label", "text"]].isna().sum())

print("\nNumber of classes:", df["label"].nunique())
print("Classes:", sorted(df["label"].unique()))

Columns: ['label', 'text']
Missing values:
label    0
text     0
dtype: int64

Number of classes: 8
Classes: ['Anxiety', 'Depression', 'Happy', 'Mentalhealth', 'Normal', 'Sad', 'Stress', 'Suicidal']


## 4. Remove missing or empty text

Rows without usable text cannot be used for text classification.

In [4]:
before = len(df)

df = df.dropna(subset=["label", "text"]).copy()
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"] != ""].copy()

print("Rows removed:", before - len(df))
print("Shape after text validation:", df.shape)

Rows removed: 0
Shape after text validation: (55693, 2)


## 5. Find texts with conflicting labels

If exactly the same text appears with different labels, the label is ambiguous for a text-classification experiment. These text groups will be removed.

In [5]:
label_counts_per_text = df.groupby("text")["label"].nunique()
conflicting_texts = label_counts_per_text[label_counts_per_text > 1].index

print("Number of conflicting text groups:", len(conflicting_texts))

if len(conflicting_texts) > 0:
    print("Rows belonging to conflicting groups:", df["text"].isin(conflicting_texts).sum())

Number of conflicting text groups: 47
Rows belonging to conflicting groups: 100


## 6. Remove conflicting text groups

In [6]:
before = len(df)

df = df[~df["text"].isin(conflicting_texts)].copy()

print("Rows removed:", before - len(df))
print("Shape after removing conflicts:", df.shape)

Rows removed: 100
Shape after removing conflicts: (55593, 2)


## 7. Remove exact duplicate texts

After removing conflicting groups, identical texts with the same label are duplicates. We keep one copy so that the same text cannot appear in multiple splits.

In [7]:
before = len(df)

df = df.drop_duplicates(subset=["text"], keep="first").copy()
df = df.reset_index(drop=True)

print("Duplicate rows removed:", before - len(df))
print("Final prepared dataset shape:", df.shape)

Duplicate rows removed: 1143
Final prepared dataset shape: (54450, 2)


## 8. Check the prepared class distribution

In [8]:
class_distribution = df["label"].value_counts().to_frame("count")
class_distribution["percentage"] = (
    class_distribution["count"] / len(df) * 100
)

print(class_distribution)

              count  percentage
label                          
Normal        16031   29.441690
Depression    15830   29.072544
Suicidal      11455   21.037649
Anxiety        4575    8.402204
Stress         2291    4.207530
Mentalhealth   1945    3.572084
Sad            1639    3.010101
Happy           684    1.256198


## 9. Create a fixed stratified train/validation/test split

We use 70% for training, 15% for validation, and 15% for testing.

The test set is kept fixed and will be used by every model.

In [9]:
RANDOM_STATE = 42

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=RANDOM_STATE
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=RANDOM_STATE
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Training set:", train_df.shape)
print("Validation set:", val_df.shape)
print("Test set:", test_df.shape)

Training set: (38115, 2)
Validation set: (8167, 2)
Test set: (8168, 2)


## 10. Verify class distribution in each split

In [10]:
print("Training distribution")
print(train_df["label"].value_counts(normalize=True).sort_index())

print("\nValidation distribution")
print(val_df["label"].value_counts(normalize=True).sort_index())

print("\nTest distribution")
print(test_df["label"].value_counts(normalize=True).sort_index())

Training distribution
label
Anxiety         0.084009
Depression      0.290725
Happy           0.012567
Mentalhealth    0.035708
Normal          0.294425
Sad             0.030093
Stress          0.042083
Suicidal        0.210390
Name: proportion, dtype: float64

Validation distribution
label
Anxiety         0.083997
Depression      0.290682
Happy           0.012612
Mentalhealth    0.035754
Normal          0.294355
Sad             0.030121
Stress          0.042121
Suicidal        0.210359
Name: proportion, dtype: float64

Test distribution
label
Anxiety         0.084109
Depression      0.290769
Happy           0.012488
Mentalhealth    0.035749
Normal          0.294442
Sad             0.030118
Stress          0.041993
Suicidal        0.210333
Name: proportion, dtype: float64


## 11. Check for text leakage between splits

There should be no identical text shared between train, validation, and test sets.

In [11]:
train_texts = set(train_df["text"])
val_texts = set(val_df["text"])
test_texts = set(test_df["text"])

print("Train ∩ Validation:", len(train_texts & val_texts))
print("Train ∩ Test:", len(train_texts & test_texts))
print("Validation ∩ Test:", len(val_texts & test_texts))

assert len(train_texts & val_texts) == 0
assert len(train_texts & test_texts) == 0
assert len(val_texts & test_texts) == 0

print("No text leakage detected.")

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0
No text leakage detected.


## 12. Save the prepared splits

The original dataset is not modified. The experimental splits are saved in `data/prepared/`.

In [12]:
output_dir = DATA_PATH.parent / "prepared"
output_dir.mkdir(parents=True, exist_ok=True)

train_df.to_csv(output_dir / "train.csv", index=False)
val_df.to_csv(output_dir / "validation.csv", index=False)
test_df.to_csv(output_dir / "test.csv", index=False)

print("Saved files:")
print(output_dir / "train.csv")
print(output_dir / "validation.csv")
print(output_dir / "test.csv")

Saved files:
..\data\prepared\train.csv
..\data\prepared\validation.csv
..\data\prepared\test.csv


## 13. Final summary

In [13]:
print("Final dataset preparation summary")
print("---------------------------------")
print("Original rows:", original_rows)
print("Prepared rows:", len(df))
print("Training rows:", len(train_df))
print("Validation rows:", len(val_df))
print("Test rows:", len(test_df))
print("Number of classes:", df["label"].nunique())
print("Random state:", RANDOM_STATE)

print("\nDataset preparation completed successfully.")

Final dataset preparation summary
---------------------------------
Original rows: 55693
Prepared rows: 54450
Training rows: 38115
Validation rows: 8167
Test rows: 8168
Number of classes: 8
Random state: 42

Dataset preparation completed successfully.
